# Multitemporal and multifactor probabilistic route modelling

This notebook has two main sections:<br>
    - The first, dedicated to the generation of cost surfaces in GEE (cells 1-4).<br>
    - The second, dedicated to the generation of the probabilistic corridors (cells 5-12).<br>
<br>
Instructions are provided as markdown cells but also within the code, be sure to follow them for an efficient use.

## Google Earth Engine (GEE)-based multitemporal multifactor cost surfaces
**Behavior:** This notebook **forces a fresh OAuth flow every run** (no cached tokens), so it’s safe to share and run on different machines.

**How to run**
1. Run **Cell 1** to install exact-range versions (once per machine or after kernel reset).
2. Run **Cell 2** to check versions.
3. Run **Cell 3** to authenticate (you’ll be prompted) and initialize EE (optionally provide a Cloud Project ID).
4. Run **Cell 4** for the generation of the cost surface. In the ouput map, export the result to your Google Drive, so it can be computed using Google Cloud resources.

**Notes**
- Tokens are wiped from typical EE cache folders at the start of each run.
- In Colab, auth is ephemeral; this notebook still enforces a fresh flow.
- If you use a Google Cloud project with billing/exports, have the Project ID ready.


In [ ]:
# Installs with compatibility pins
import sys, subprocess, importlib

PKGS = [
    "earthengine-api==1.6.15",
    "geemap==0.36.6",
    "google-auth==2.41.1",
    "google-auth-oauthlib==1.2.3",
    "google-cloud-storage==3.4.1",
    "ipykernel==6.25.0",
    "ipython==8.15.0",
    "ipywidgets==8.0.4",
    "jupyterlab==4.4.10",
    "numpy==1.26.2",
    "rasterio==1.3.9",
]

def ensure(pkgs):
    # Install missing or unpinned packages into the current environment/kernel
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--quiet", *pkgs]
    print("Installing/validating packages…")
    subprocess.check_call(cmd)

try:
    import ee, geemap  # quick check; if this fails, we install everything
except Exception:
    ensure(PKGS)

# Write a requirements lock (range-pinned) alongside the notebook
with open("requirements_M2PRM.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(PKGS) + "\n")

print("✔ Environment ready. Wrote 'requirements_M2PRM.txt' with compatibility pins.")


In [ ]:
# Comprehensive environment check
from importlib import import_module
from importlib.metadata import version as dist_version, PackageNotFoundError
import re

# 1) Use the PKGS list from the first cell (strings like "numpy==1.26.2")
pins = {}
for spec in PKGS:
    m = re.match(r"^\s*([A-Za-z0-9_.\-]+)\s*==\s*([^\s]+)\s*$", spec)
    if m:
        pins[m.group(1)] = m.group(2)

# 2) Map distribution names to importable module paths
name_map = {
    "earthengine-api": "ee",
    "geemap": "geemap",
    "google-auth": "google.auth",
    "google-auth-oauthlib": "google_auth_oauthlib",
    "google-auth-httplib2": "google_auth_httplib2",
    "google-api-python-client": "googleapiclient",
    "google-cloud-storage": "google.cloud.storage",
    "numpy": "numpy",
    "rasterio": "rasterio",
    # add more if you pin them
}

def check(dist_name, mod_path):
    try:
        mod = import_module(mod_path)
        try:
            installed = dist_version(dist_name)
        except PackageNotFoundError:
            # fallback: try module __version__
            installed = getattr(mod, "__version__", "?")
        pinned = pins.get(dist_name)
        status = "OK" if (pinned and installed == pinned) else ("MISMATCH" if pinned else "UNPINNED")
        print(f"{dist_name:<25} import={mod_path:<24} version={installed:<12} pin={pinned or '-':<10} [{status}]")
    except Exception as e:
        print(f"{dist_name:<25} import={mod_path:<24} ERROR: {type(e).__name__}: {e}")

for dist, mod in name_map.items():
    if dist in pins:
        check(dist, mod)

# Optional: quick Python version echo
import sys
print(f"\nPython: {sys.version.split()[0]}")


In [ ]:
# Fresh EE OAuth flow on every run
import os, shutil
from pathlib import Path

def wipe_ee_tokens():
    candidates = [
        Path.home() / ".config" / "earthengine",                    # Linux/macOS
        Path.home() / "AppData" / "Roaming" / "earthengine",        # Windows
    ]
    for d in candidates:
        if d.exists():
            try:
                shutil.rmtree(d, ignore_errors=True)
            except Exception as e:
                print(f"Warning: could not remove {d}: {e}")

wipe_ee_tokens()

import ee

# Notebook/CLI-friendly OAuth. This will open a link or prompt inline (e.g., Colab).
ee.Authenticate(auth_mode="notebook")

# Optional: Ask for a Google Cloud Project ID (needed for some exports/quotas)
PROJECT_ID = os.environ.get("EE_PROJECT") or input(
    "EE Cloud Project ID (press Enter to skip): "
).strip() or None

# Initialize EE
if PROJECT_ID:
    ee.Initialize(project=PROJECT_ID)
else:
    ee.Initialize()

# Quick sanity check (forces a tiny API call)
print("User profile email:", ee.data.getAssetRoots()[0]["id"].split(":")[0] if ee.data.getAssetRoots() else "(no asset roots)")
print("EE initialized ✔")


In [ ]:
# Cost surface map: UI + compute + preview + export (Drive / Local)

import ee, geemap, math, os, time, io, requests
from IPython.display import display
import ipywidgets as W

# UI controls
month_choices = [
    ("Whole year", "yr"), ("Jan", "jan"), ("Feb", "feb"), ("Mar", "mar"),
    ("Apr", "apr"), ("May", "may"), ("Jun", "jun"), ("Jul", "jul"),
    ("Aug", "aug"), ("Sep", "sep"), ("Oct", "oct"), ("Nov", "nov"), ("Dec", "dec")
]
month = W.Dropdown(options=month_choices, value="jan", description="Period:", layout=W.Layout(width="260px"))

max_cost = W.BoundedFloatText(value=40.0, min=1, max=9999, step=1.0, description="Max cost:", layout=W.Layout(width="260px"))
bit_depth = W.RadioButtons(options=[("Float (recommended)", "Float"), ("8-bit", "8-bit")], value="Float", description="Depth:")
use_water_attraction = W.ToggleButtons(options=[("No", "n"), ("Yes", "y")], value="n", description="Water attract.:")
out_res = W.Text(value="", description="Out res (m):", placeholder="blank/0 = DSM native", layout=W.Layout(width="260px"))
out_res_hint = W.HTML("<div style='margin-top:-10px;font-size:10px;color:#666'>blank or 0 = DSM native resolution</div>")

export_target = W.Dropdown(options=[("Google Drive", "drive"), ("None (preview only)", "none")],
                           value="none", description="Export to:", layout=W.Layout(width="260px"))
export_name = W.Text(value="costsurf_", description="Name:", layout=W.Layout(width="300px"))

run_btn = W.Button(description="Build, Preview & (Optionally) Export", button_style="primary", icon="play")
status = W.HTML()

controls = W.HBox([
    W.VBox([month, max_cost, bit_depth]),
    W.VBox([use_water_attraction, out_res, out_res_hint, export_target]),
    W.VBox([export_name, run_btn, status]),
])

display(controls)

# Geometry & base. Modify as necessary but make sure you use a loose sand map that covers your study area if you intend to use this cost
geometry = ee.Geometry.Polygon([
    [26.034856936314327, 31.965400777310872],
    [34.19255939893447, 27.520378920323584],
    [70.85299048338345, 19.854238685800006],
    [90.28374707240327, 20.074324423110866],
    [112.29527848792249, 21.22664649657822],
    [112.23560562718355, 47.3578548447329],
    [70.6364691148242, 53.122300330081565],
    [25.974978246994738, 47.85290888472198],
    [26.034856936314327, 31.965400777310872]
])

Map = geemap.Map(center=[37.566, 67.601], zoom=5)  # change as necessary
Map.setCenter(67.601, 37.566, 5)
display(Map)

# Helpers
def _period_from_choice(choice):
    # Returns (ee.Filter.dayOfYear, monthNum, altNum, periodName)
    per_map = {
        "yr":  (ee.Filter.dayOfYear(1, 364), None, None, "yr"),
        "jan": (ee.Filter.dayOfYear(1, 31), "00", "01", "jan"),
        "feb": (ee.Filter.dayOfYear(32, 59), "01", "02", "feb"),
        "mar": (ee.Filter.dayOfYear(60, 90), "02", "03", "mar"),
        "apr": (ee.Filter.dayOfYear(91, 120), "03", "04", "apr"),
        "may": (ee.Filter.dayOfYear(121, 151), "04", "05", "may"),
        "jun": (ee.Filter.dayOfYear(152, 181), "05", "06", "jun"),
        "jul": (ee.Filter.dayOfYear(182, 212), "06", "07", "jul"),
        "aug": (ee.Filter.dayOfYear(213, 243), "07", "08", "aug"),
        "sep": (ee.Filter.dayOfYear(244, 273), "08", "09", "sep"),
        "oct": (ee.Filter.dayOfYear(274, 304), "09", "10", "oct"),
        "nov": (ee.Filter.dayOfYear(305, 334), "10", "11", "nov"),
        "dec": (ee.Filter.dayOfYear(335, 365), "11", "12", "dec"),
    }
    return per_map[choice]

def _albedo_palette():
    # Palette (blue→green→yellow→orange→red→magenta)
    return ['0f00ff', '03ff00', 'efff00', 'ffbc00', 'ff0000', 'ff0081']

def _js_round_positive(value):
    return int(math.floor(float(value) + 0.5))

# Core compute
def build_cost_surface(periodSel, maxCost, bitSel, queryConv, outrr_override):
    maxCost = _js_round_positive(maxCost)

    # DSM: ALOS AW3D30 V4.1, filtered to the study area geometry
    aw3d = ee.ImageCollection('JAXA/ALOS/AW3D30/V4_1').filterBounds(geometry)
    elevation = aw3d.select('DSM')
    proj = elevation.first().select(0).projection()
    dsm = elevation.mosaic().setDefaultProjection(proj).clip(geometry)

    # Period
    period, monthNum, altNum, periodName = _period_from_choice(periodSel)

    # Slope (in degrees → radians → tan)
    slope = ee.Terrain.slope(dsm)
    slp_rad = slope.multiply(3.1415926536).divide(180)
    slp_dg = slp_rad.tan()

    # Herzog/Minetti cost of slope
    costSlp1 = (slp_dg.pow(6).multiply(1337.8)
                .add(slp_dg.pow(5).multiply(278.19))
                .subtract(slp_dg.pow(4).multiply(517.39))
                .subtract(slp_dg.pow(3).multiply(78.199))
                .add(slp_dg.pow(2).multiply(93.419))
                .add(slp_dg.multiply(19.825))
                .add(1.64))
    costSlp2 = costSlp1.select(['slope'], ['cost'])
    costSlp = (costSlp2.gt(99.99).multiply(99.99)
               .add(costSlp2.lte(99.99).multiply(costSlp2)))

    # Snow (MODIS daily → mean over period)
    snowBase = (ee.ImageCollection('MODIS/061/MOD10A1').select('NDSI_Snow_Cover')
                .filter(period)
                .filterBounds(geometry)
                .reduce(ee.Reducer.mean())
                .unmask(0)
                .clip(geometry))
    snow = snowBase.divide(50).add(ee.Image(1))  #  values from 1 to 3

    # Sea mask: AW3D30 V4.1 MSK sea cells + independent Caspian Sea mask
    mskALOS = (aw3d
               .select('MSK')
               .mosaic()
               .clip(geometry)
               .setDefaultProjection(proj))

    alosSea = (mskALOS  # In the AW3D MSK band, cells classified as sea have a value of 3
               .bitwiseAnd(3)
               .eq(3)
               .unmask(0)
               .toByte()
               .rename('allSeas'))

    caspianRegion = ee.Geometry.Rectangle([46.0, 36.0, 55.2, 47.8], None, False)

    caspianSea = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
                  .select('occurrence')
                  .clip(caspianRegion)
                  .unmask(0)
                  .gte(90)
                  .toByte()
                  .rename('allSeas'))

    allSeas = ee.ImageCollection([alosSea, caspianSea]).mosaic()
    seaMsk = allSeas.multiply(maxCost).float()

    # Dams/Reservoirs (imported asset)
    maskDams = (ee.FeatureCollection('users/hao23/Grand_reservoirs')
                .reduceToImage(properties=['value'], reducer=ee.Reducer.first())
                .unmask(0)
                .remap([0, 1], [1, 0])
                .clip(geometry))
    dams = maskDams.remap([0, 1], [3.28, 0]).float()

    # Cold: WorldClim daytime temperature proxy + TerraClimate wind chill
    tempMinCold, tempMaxCold = -13, 0

    windSpeed = (ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').select('vs')
                 .filter(period)
                 .filterBounds(geometry)
                 .reduce(ee.Reducer.mean())
                 .unmask(0)
                 .multiply(0.036)
                 .clip(geometry))  # scale factor 0.01 and m/s → km/h (×3.6)

    # Daytime temperature proxy: intermediate between the 24-hour monthly mean and the monthly average daily maximum
    if periodSel == 'yr':
        temp_bands = (ee.ImageCollection('WORLDCLIM/V1/MONTHLY')
                .select(['tavg', 'tmax'])
                .mean()
                .multiply(0.1))  # WorldClim temperatures are stored as °C * 10
        
        temp = (temp_bands.select('tavg')
                .add(temp_bands.select('tmax'))
                .divide(2)
                .rename('tday_proxy'))

    else:

        temp_bands = (ee.Image(f'WORLDCLIM/V1/MONTHLY/{altNum}')
                .select(['tavg', 'tmax'])
                .multiply(0.1))  # WorldClim temperatures are stored as °C * 10

        temp = (temp_bands.select('tavg')
                .add(temp_bands.select('tmax'))
                .divide(2)
                .rename('tday_proxy'))

    wcValues = (ee.Image(13.12)
                .add(ee.Image(0.6215).multiply(temp))
                .subtract(ee.Image(11.37).multiply(windSpeed.pow(0.16)))
                .add(ee.Image(0.3965).multiply(temp).multiply(windSpeed.pow(0.16))))

    windChill = (temp.lte(10).And(windSpeed.gte(4.8)).multiply(wcValues)
                 .add(temp.gt(10).Or(windSpeed.lt(4.8)).multiply(temp)))

    coldMultiplier = 1.5
    
    cold1 = windChill.gte(tempMinCold)
    cold2 = windChill.lte(tempMaxCold)
    cold3 = windChill.lt(tempMinCold).multiply(tempMinCold)
    cold4 = windChill.gt(tempMaxCold).multiply(tempMaxCold)
    cold5 = cold1.multiply(cold2).multiply(windChill).add(cold3).add(cold4)
    cold6 = cold5.subtract(tempMinCold).divide(tempMaxCold - tempMinCold)
    cold = (ee.Image(coldMultiplier).subtract(cold6.multiply(coldMultiplier - 1)))  # values ranging from 1 to 1.5

    # Loose sand / dunes (imported asset. Expects a probability map)
    sandProb = ee.Image('users/hao23/looseSand_SA').unmask(0).clip(geometry)  # Use 'users/hao23/looseSand_RE' for the Roman Empire area
    looseSand = ee.Image(1).add(sandProb.gte(0.7).multiply(sandProb.multiply(0.35)))

    # Lack of water: same WorldClim daytime proxy, 26..38°C → 0 to 2 heat exponent
    tempMin, tempMax = 26, 38

    heat1 = temp.gte(tempMin)
    heat2 = temp.lte(tempMax)
    heat3 = temp.lt(tempMin).multiply(tempMin)
    heat4 = temp.gt(tempMax).multiply(tempMax)
    heat5 = heat1.multiply(heat2).multiply(temp).add(heat3).add(heat4)
    heat = heat5.subtract(tempMin).divide(tempMax - tempMin).multiply(2)  # values from 0 to 2

    # EVI (multi-annual / extended window)
    if periodSel == 'yr':
        evi = (ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_8DAY_EVI')
               .filterDate('1984-01-01', '1995-12-31')
               .filter(period)
               .filterBounds(geometry)
               .reduce(ee.Reducer.mean())
               .unmask(0)
               .clip(geometry))
    else:
        evi = (ee.ImageCollection('LANDSAT/COMPOSITES/C02/T1_L2_8DAY_EVI')
               .filterDate('1984-01-01', '2015-12-31')
               .filter(period)
               .filterBounds(geometry)
               .reduce(ee.Reducer.mean())
               .unmask(0)
               .clip(geometry))

    # Desert factor via EVI threshold: 1..2, with negative EVI treated as water (=1)
    dsrtThrs = 0.075
    desert1 = evi.gte(0)
    desert2 = evi.lte(dsrtThrs)
    desert3 = evi.lt(0)
    desert4 = evi.gt(dsrtThrs).multiply(dsrtThrs)
    desert5 = desert1.multiply(desert2).multiply(evi).add(desert4)
    desert6 = desert5.divide(dsrtThrs).add(1)
    desert = desert6.subtract(3).multiply(-1).subtract(desert3)

    # Raw aridity–heat index: values from 1 to 4
    noWatRaw = desert.pow(heat)

    # Rescale raw index to the final conservative multiplier: values from 1 to 1.5
    rawMaximum = 2 ** 2  # maximum desert index (2) raised to maximum heat exponent (2)
    noWaterMaximum = 1.5

    noWat = (noWatRaw
             .subtract(1)
             .divide(rawMaximum - 1)
             .multiply(noWaterMaximum - 1)
             .add(1)
             .clamp(1, noWaterMaximum))

    # Water attraction (optional, heavy. Use only if necessary)
    if queryConv == 'y':
        rWat = 15
        watConv = (noWat
                   .gt(1)
                   .focal_median(kernel=ee.Kernel.circle(radius=(rWat * 1000), units='meters')))
        waterThrs = 0.2
        watAtt = watConv.multiply(evi.gte(waterThrs)).remap([0, 1], [1, 0.75])
    else:
        watAtt = ee.Image(1)

    # Surface water (JRC)
    if periodSel == 'yr':
        watRecur = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
        surfWat = watRecur.divide(33.33).add(ee.Image(1)).clip(geometry)
    else:
        watRecur = ee.Image('JRC/GSW1_4/MonthlyRecurrence/monthly_recurrence_' + monthNum).select('monthly_recurrence').unmask(0)
        surfWat = watRecur.divide(33.33).add(ee.Image(1)).clip(geometry)

    # Altitude penalty above 2000 m
    height = (ee.Image(-0.0170146)
              .subtract(ee.Image(0.004039874802622)
                        .multiply(ee.Image(1)
                                  .subtract(ee.Image(2.718281828459)
                                            .pow(dsm.multiply(ee.Image(0.0008254486)))))))
    height = height.multiply(dsm.gt(2000)).add(ee.Image(1))

    # Raw combined cost
    costsRaw = (costSlp
                .multiply(noWat)
                .multiply(watAtt)
                .multiply(surfWat)
                .multiply(snow)
                .multiply(looseSand)
                .multiply(cold)
                .multiply(height)
                .multiply(maskDams).add(dams)
                .add(seaMsk))

    # Cap at maxCost, then scale
    costsMax = (costsRaw.gt(maxCost).multiply(maxCost)
                .add(costsRaw.lte(maxCost).multiply(costsRaw)))

    if bitSel in ("8-bit", "8"):
        costs = costsMax.divide(maxCost).multiply(255).byte().clip(geometry).unmask(255)
        vis = dict(min=1, max=150, palette=_albedo_palette())
    else:
        costs = (costsMax
                 .multiply(ee.Number(10).pow(4))
                 .round()
                 .divide(ee.Number(10).pow(4))
                 .multiply(0.0001)
                 .clip(geometry)
                 .unmask(0.004))
        vis = dict(min=0.0001, max=0.001, palette=_albedo_palette())

    # Output resolution: rounding of the DSM nominal scale
    rr = round(float(proj.nominalScale().getInfo()) * 10000) / 10000
    if (outrr_override is None) or (float(outrr_override) <= rr):
        outrr = rr
    else:
        outrr = float(outrr_override)

    return dict(
        image=costs.rename('cost'),
        vis=vis,
        periodName=periodName,
        scale=outrr,
        proj=proj
    )

def add_preview_layer(costs, vis):
    # Clear & add layer
    Map.clear_layers()
    Map.addLayer(costs, vis, "Cost surface")
    Map.centerObject(geometry, 5)

def snapshot_png(costs, vis, fname="cost_surface_preview.png"):
    # Thumbnail snapshot for the notebook output
    url = costs.visualize(**vis).getThumbURL({
        "region": geometry,
        "dimensions": 1024,
        "format": "png"
    })
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(fname, "wb") as f:
            f.write(r.content)
        from IPython.display import Image
        display(Image(filename=fname))
        return fname
    except Exception as e:
        print("Snapshot failed:", e)
        return None

def export_drive(costs, name, scale):
    task = ee.batch.Export.image.toDrive(
        image=costs,
        description=name,
        region=geometry,
        scale=scale,
        maxPixels=9e12
    )
    task.start()
    return task.id

def export_local(costs, name, scale):
    # Download GeoTIFF directly to the notebook machine
    out_tif = f"{name}.tif"
    geemap.ee_export_image(
        image=costs, filename=out_tif,
        scale=scale, region=geometry, file_per_band=False, crs="EPSG:4326"
    )
    return out_tif

# Wire UI
def _run_clicked(_):
    status.value = "<b>Running…</b>"
    try:
        cfg = build_cost_surface(
            periodSel=month.value,
            maxCost=float(max_cost.value),
            bitSel=bit_depth.value,
            queryConv=use_water_attraction.value,
            outrr_override=(None if str(out_res.value).strip() in ["", "0"] else float(out_res.value))
        )
        costs, vis = cfg["image"], cfg["vis"]
        add_preview_layer(costs, vis)
        snap = snapshot_png(costs, vis, fname=f"{export_name.value}_preview.png")

        # Export
        if export_target.value == "drive":
            tid = export_drive(costs, f"{export_name.value}_{cfg['periodName']}_{int(round(cfg['scale']))}m", cfg["scale"])
            status.value = f"✅ Done. Snapshot shown. Drive export task id: <code>{tid}</code>."
        else:
            status.value = "✅ Done. Snapshot shown. (No export requested.)"

    except Exception as e:
        status.value = f"<span style='color:red'>❌ Error: {e}</span>"

run_btn.on_click(_run_clicked)

## GRASS GIS-based raster analysis for the generation of probabilistic corridors
**Behavior:** The following cells need GRASS GIS to be installed in a local machine to work.

**How to run**
1. Run **Cell 1** to set up paramenters and environment providing:
  - A path to the cost surface raster generated by the execution of the previous cell (donloaded from Google Drive).
  - The number of origins/destinations for the corridors.
  - The coordinates (longitude,latitute) of origins/destinations use the same CRS of the cost surface raster WGS84 (EPSG 4326).
2. Run **Cell 2** to provide the path to GRASS' bat file and to check its version.
3. Run **Cell 3** to generate the cost distance rasters.
4. Run **Cell 4** to define the pairs of cost distance rasters (generated during the execution of the preivous cell) taht will form the corridors.
5. Run **Cell 5** to compute the corridors.
6. Run **Cell 6** to make sure the corridors created in Step 5 are avialable.
7. Run **Cell 7** to normalise corridors to 0–100 as Float64.
8. Run **Cell 8** to create a minimum raster composite over all normalized corridors producing the final output of the code.

**Notes**
- Please make sure all data needed is provided within the code.
- The code paralellizes the generation of outputs. Make sure you have plenty of computational resources available: as many cores as origin/destinations and large RAM memory. If this is not the case consider to run the algorithm in a server or run the processes sequentially.



In [ ]:
# Step 1: Parameters (run first)

from pathlib import Path
import rasterio as rio
import numpy as np

# 1) Path to your cost surface raster (e.g., r"C:\data\cost_surface.tif")
COST_RASTER_PATH = r"/home/giap-clab-2/Desktop/m2conet/rasters_31Jul26/multiple_routes/costsurf_SilkRoute_31Jul26_yrAVG_618m.tif"  # <-- fill in

# 2) How many cost-distance rasters to compute?
N_OUTPUTS = 4  # <-- set an integer >= 1

# 3) r.cost memory in MB (per concurrent r.cost process)
#    Be sure N_WORKERS * RCOST_MEMORY_MB fits comfortably within your available RAM.
RCOST_MEMORY_MB = 45000  # e.g. 8192 = 8 GB, 16384 = 16 GB

# 4) Start coordinates (Easting/X, Northing/Y) in the SAME CRS as COST_RASTER_PATH.
#    Provide exactly N_OUTPUTS pairs. Example:
#    START_COORDS = [(508450.0, 4641230.0), (512000.0, 4639000.0), (515000.0, 4635000.0)]
START_COORDS = [(29.019785,41.022583),(108.947162,34.264356),(66.90118,36.76792),(91.83598,22.3278)]  # <-- fill in with a list of (x, y) coordinates

# --------------------------------------------------------------------------
# Basic validation
# --------------------------------------------------------------------------
assert COST_RASTER_PATH, "Please provide COST_RASTER_PATH."
cost_path = Path(COST_RASTER_PATH)
assert cost_path.exists(), f"Cost raster not found: {cost_path}"

assert isinstance(N_OUTPUTS, int) and N_OUTPUTS >= 1, "N_OUTPUTS must be an integer >= 1."
assert isinstance(START_COORDS, (list, tuple)) and len(START_COORDS) == N_OUTPUTS, \
    "START_COORDS must be a list/tuple with exactly N_OUTPUTS (x, y) pairs."

# Read CRS and stats we will use to set null_cost
with rio.open(cost_path) as src:
    crs = src.crs
    epsg = crs.to_epsg() if crs is not None else None

# Compute a conservative null_cost (max of valid pixels) so NULLs are traversable but expensive
# If the raster is large, this single pass is still cheaper than doing it in each process.
data = None
with rio.open(cost_path) as src:
    # Read in manageable chunks if needed; here we try a single read
    data = src.read(1, masked=True)
    # If *all* values are masked (unlikely), fallback to 1.0
    if np.ma.count(data) == 0:
        NULL_COST = 1.0
    else:
        NULL_COST = float(0.004)  # Use 0.004 if a 16bit cost surface is used and 255 if using a 8bit

print("✓ Inputs OK")
print(f"Cost raster: {cost_path}")
print(f"CRS: {crs}  |  EPSG: {epsg if epsg else 'unknown (XY tmp-location will be used)'}")
print(f"Number of outputs: {N_OUTPUTS}")
print(f"Start coords: {START_COORDS}")
print(f"null_cost to use: {NULL_COST}")
print(f"r.cost memory: {RCOST_MEMORY_MB} MB")


In [ ]:
# Step 2: Checking GRASS availability and version

from pathlib import Path
import shutil, subprocess, sys

# EDIT this to your actual GRASS .bat path (examples below)
# GRASS_BIN = r"C:\Program Files\GRASS GIS 8.4\grass84.bat"
# GRASS_BIN = r"C:\OSGeo4W\bin\grass.bat"
GRASS_BIN = r"/usr/bin/grass"  # <-- fill in

assert GRASS_BIN, "Set GRASS_BIN to your GRASS launcher .bat file."
print("Using GRASS_BIN:", GRASS_BIN)

# Version check
try:
    out = subprocess.run([GRASS_BIN, "--version"], capture_output=True, text=True, check=True)
    print(out.stdout.strip() or out.stderr.strip())
except Exception as e:
    print("GRASS check failed:", e, file=sys.stderr)
    raise


In [ ]:
# Step 3: r.cost execution

import os
import shlex
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

out_dir = cost_path.parent
base = cost_path.stem

def q_path(p):
    """OS-aware quoting for file paths used inside shell commands."""
    p = str(p)
    if os.name == "nt":
        return f'"{p}"' if (" " in p or "(" in p or ")" in p) else p
    else:
        return shlex.quote(p)

def build_grass_pipeline(x, y, out_tif, epsg_code, null_cost, in_path):
    """
    Build command to run GRASS modules in a temporary location, with OS-specific chaining.
    """
    in_q  = q_path(in_path)
    out_q = q_path(out_tif)

    cmds = [
        f"r.in.gdal input={in_q} output=cost --o",
        "g.region raster=cost",
        f"r.cost -k input=cost start_coordinates={x},{y} "
        f"output=cd_out null_cost={null_cost} memory={RCOST_MEMORY_MB}",
        (
            "r.out.gdal "
            f"input=cd_out output={out_q} format=GTiff type=Float64 "
            'createopt="COMPRESS=LZW,TILED=YES,BIGTIFF=IF_SAFER" --o'
        ),
    ]
    tmp_loc = f"EPSG:{epsg_code}" if epsg_code else "XY"

    base_cmd = [GRASS_BIN, "--tmp-location", tmp_loc, "--exec"]

    if os.name == "nt":  # Windows
        chain = " && ".join(cmds)
        shell_part = ["cmd", "/c", chain]
    else:                # Linux/macOS
        chain = " ; ".join(cmds)
        shell_part = ["bash", "-lc", chain]

    return base_cmd + shell_part

def run_one(idx_coord):
    idx, (x, y) = idx_coord
    out_name = f"{base}_costdist_{idx+1}.tif"
    out_tif = out_dir / out_name
    cmd = build_grass_pipeline(x, y, out_tif, epsg, NULL_COST, cost_path)
    try:
        res = subprocess.run(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True
        )
        return (out_tif, res.stdout.strip(), res.stderr.strip(), True)
    except subprocess.CalledProcessError as e:
        return (out_tif, e.stdout, e.stderr, False)
    except Exception as e:
        return (out_tif, "", str(e), False)

max_workers = min(N_OUTPUTS, (os.cpu_count() or 1))
print(f"Launching {N_OUTPUTS} r.cost jobs with max_workers={max_workers} (ThreadPool)...")

results = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(run_one, (idx, xy)) for idx, xy in enumerate(START_COORDS)]
    for fut in as_completed(futures):
        results.append(fut.result())

ok, failed = 0, 0
for out_tif, sout, serr, success in results:
    if success and out_tif.exists():
        ok += 1
        print(f"✓ Wrote {out_tif}")
    else:
        failed += 1
        print(f"✗ Failed for {out_tif.name}\nSTDOUT:\n{sout}\nSTDERR:\n{serr}\n")

print(f"\nDone. Successful: {ok} | Failed: {failed}")
print(f"Outputs directory: {out_dir}")


In [ ]:
# Step 4: Corridor pairs selection

from pathlib import Path
import re

# If you ran Step 1, these already exist:
try:
    out_dir
    base
except NameError:
    # Fallback if running standalone: infer from COST_RASTER_PATH
    from pathlib import Path
    assert COST_RASTER_PATH, "Please set COST_RASTER_PATH as in Step 1."
    cost_path = Path(COST_RASTER_PATH)
    out_dir = cost_path.parent
    base = cost_path.stem

# Discover cost-distance rasters created in Step 1
cd_files = sorted(out_dir.glob(f"{base}_costdist_*.tif"))

assert cd_files, f"No cost-distance rasters found in {out_dir}. Expected files like '{base}_costdist_1.tif'."

print("Found cost-distance rasters:")
for i, p in enumerate(cd_files, 1):
    print(f"  {i:2d}: {p.name}")

# --------------------------------------------------------------------
# Define the PAIRS to average (by 1-based indices from the above list)
# Example: PAIRS = [(1,2), (1,3), (2,4)]
# --------------------------------------------------------------------
PAIRS = [(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)]  # <-- fill in your pairs, e.g., [(1,2), (3,5)]


In [ ]:
# Step 5: Corridor computation

import os, subprocess, sys, platform, ctypes, tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Expect base, out_dir, PAIRS from previous cells

assert cd_files, "No cost-distance rasters (cd_files) found."
assert isinstance(PAIRS, (list, tuple)) and all(isinstance(p, (list, tuple)) and len(p) == 2 for p in PAIRS), \
       "PAIRS must be a list of (i, j) tuples."

n = len(cd_files)
pairs_0 = []
for (i, j) in PAIRS:
    assert 1 <= i <= n and 1 <= j <= n and i != j, f"Invalid pair {(i, j)}. Indices must be 1..{n} and different."
    pairs_0.append((i-1, j-1))  # 0-based

# Prefer EPSG tmp-location if available; fallback to XY
try:
    tmp_loc = f"EPSG:{epsg}" if epsg else "XY"
except NameError:
    tmp_loc = "XY"

IS_WINDOWS = platform.system().lower().startswith("win")

# ---- Windows path handling: no quotes; use 8.3 if spaces/parentheses ----
def win_short_path(path_str: str) -> str:
    buf = ctypes.create_unicode_buffer(32768)
    GetShortPathNameW = ctypes.windll.kernel32.GetShortPathNameW
    ret = GetShortPathNameW(path_str, buf, len(buf))
    return buf.value if ret else path_str

def safe_arg_path(p: Path) -> str:
    s = str(p)
    if IS_WINDOWS:
        if any(c in s for c in " ()"):
            s = win_short_path(s)
        return s  # unquoted
    else:
        return "'" + s.replace("'", "'\\''") + "'"

def build_grass_command_chain(in_a: Path, in_b: Path, out_tif: Path):
    a = safe_arg_path(in_a.resolve())
    b = safe_arg_path(in_b.resolve())
    o = safe_arg_path(out_tif.resolve())

    # Build r.mapcalc step:
    # - On Windows: write expression to temp file and use file=<path>
    # - On Unix: pass expression= with single quotes
    if IS_WINDOWS:
        # Create temp expression file (we return both the GRASS command and the temp path to clean later)
        tf = tempfile.NamedTemporaryFile(prefix="rmapcalc_", suffix=".txt", delete=False)
        tf.write(b"corridor = (a + b) / 2.0")
        tf.flush(); tf.close()
        expr_path = safe_arg_path(Path(tf.name))
        cmd_calc = f"r.mapcalc file={expr_path} --o"
        temp_to_cleanup = tf.name
    else:
        cmd_calc = "r.mapcalc expression='corridor = (a + b) / 2.0' --o"
        temp_to_cleanup = None

    cmd_rin_a = f"r.in.gdal input={a} output=a --o"
    cmd_rin_b = f"r.in.gdal input={b} output=b --o"
    cmd_region = "g.region raster=a,b"
    cmd_out    = (
        f"r.out.gdal input=corridor output={o} format=GTiff type=Float64 "
        f'createopt="COMPRESS=LZW,TILED=YES,BIGTIFF=IF_SAFER" --o'
    )

    if IS_WINDOWS:
        chain = " && ".join([cmd_rin_a, cmd_rin_b, cmd_region, cmd_calc, cmd_out])
        cmd = [GRASS_BIN, "--tmp-location", tmp_loc, "--exec", "cmd", "/C", chain]
    else:
        chain = " ; ".join([cmd_rin_a, cmd_rin_b, cmd_region, cmd_calc, cmd_out])
        cmd = [GRASS_BIN, "--tmp-location", tmp_loc, "--exec", "bash", "-lc", chain]

    return cmd, temp_to_cleanup

def run_pair(i0, j0):
    a = cd_files[i0]
    b = cd_files[j0]
    out_name = f"{base}_corridor_{i0+1}_{j0+1}.tif"
    out_tif = out_dir / out_name
    cmd, tmp_expr = build_grass_command_chain(a, b, out_tif)
    try:
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        ok = out_tif.exists()
        return (out_tif, ok, res.stdout.strip(), res.stderr.strip(), tmp_expr)
    except subprocess.CalledProcessError as e:
        return (out_tif, False, e.stdout, e.stderr, tmp_expr)
    except Exception as e:
        return (out_tif, False, "", f"Python-level error: {repr(e)}", tmp_expr)

# You can set MAX_WORKERS=1 if you prefer serial execution on Windows
MAX_WORKERS = min(len(pairs_0), os.cpu_count() or 1)
print(f"Building {len(pairs_0)} corridor raster(s) with max_workers={MAX_WORKERS} (threaded) ...")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(run_pair, i0, j0) for (i0, j0) in pairs_0]
    for f in as_completed(futs):
        results.append(f.result())

# Cleanup temp files (Windows) and report
ok, failed = 0, 0
for out_tif, success, so, se, tmp_expr in results:
    if tmp_expr and os.path.exists(tmp_expr):
        try:
            os.remove(tmp_expr)
        except Exception:
            pass
    if success:
        ok += 1
        print(f"✓ Wrote {out_tif.name}")
    else:
        failed += 1
        print(f"✗ Failed: {out_tif.name}\nSTDOUT:\n{so}\nSTDERR:\n{se}\n")

print(f"\nCorridors done. Successful: {ok} | Failed: {failed}")
print(f"Directory: {out_dir}")


In [ ]:
# Step 6: Find corridor rasters created in Step 3

from pathlib import Path

# Expect: base, out_dir from previous cells
try:
    base, out_dir
except NameError:
    raise RuntimeError("Run Step 1 first to set 'base' and 'out_dir'.")

corr_files = sorted(Path(out_dir).glob(f"{base}_corridor_*_*.tif"))
assert corr_files, f"No corridor rasters found in {out_dir} with pattern '{base}_corridor_*_*.tif'."

print("Found corridor rasters:")
for p in corr_files:
    print("  ", p.name)


In [ ]:
# Step 7: Histogram-based percentile-rank normalisation (0–100, 0.001 steps, 0.001 steps) using rasterio

# Values are interpolated according to their actual position within each histogram bin, so equivalent
# valules receive equivalent ranks and no artificial spatial texture is introduced.

import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np

try:
    import rasterio
except ImportError as e:
    raise RuntimeError("Please install rasterio: pip install rasterio") from e


# Number of histogram bins used to approximate the empirical CDF.
# 100,000 retains the precision of the previous implementation.
PERCENTILE_BINS = 100_000

# Raster normalisation is both CPU- and disk-intensive. Rasterio and NumPy
# release the GIL during the expensive operations, allowing different corridor
# files to be processed concurrently. The cap prevents excessive disk I/O.
PERCENTILE_WORKERS = min(
    len(corr_files),
    max(1, min(8, (os.cpu_count() or 1) // 2)),
)

# Outside the valid 0–100 percentile range.
PERCENTILE_NODATA = np.float32(-9999.0)


def _valid_values(masked_array):
    """Return the finite, unmasked values of a Rasterio masked array."""
    mask = np.ma.getmaskarray(masked_array)
    data = np.asarray(masked_array.data, dtype=np.float64)
    valid = (~mask) & np.isfinite(data)
    return data[valid]


def _percentile_output_profile(src):
    """Create a compact Float32 GeoTIFF profile preserving georeferencing."""
    profile = src.profile.copy()

    # Remove inherited block sizes before requesting tiled output. GDAL will
    # select valid tile dimensions for the output raster.
    profile.pop("blockxsize", None)
    profile.pop("blockysize", None)

    profile.update(
        driver="GTiff",
        count=1,
        dtype="float32",
        nodata=float(PERCENTILE_NODATA),
        compress="lzw",
        predictor=3,
        tiled=True,
        BIGTIFF="IF_SAFER",
    )
    return profile


def percentile_rank_rasterio(
    in_path: Path,
    out_path: Path,
    bins: int = PERCENTILE_BINS,
) -> tuple[bool, str, str]:
    """
    Convert one corridor raster to percentile ranks from 0 to 100.

    Processing is block-based, so the complete raster is never loaded into
    memory. A high-resolution global histogram approximates the empirical CDF.
    Within each histogram bin, ranks are interpolated from the actual cell
    value rather than from cell coordinates.

    Low percentile values retain the original interpretation of high route
    potential. Input NoData cells remain NoData in the output.
    """
    in_path = Path(in_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with rasterio.Env():
            with rasterio.open(in_path, "r") as src:
                profile = _percentile_output_profile(src)

                # Pass 1: determine the global finite-data range.
                vmin = None
                vmax = None

                for _, window in src.block_windows(1):
                    values = _valid_values(
                        src.read(1, window=window, masked=True)
                    )
                    if values.size == 0:
                        continue

                    block_min = float(values.min())
                    block_max = float(values.max())

                    vmin = (
                        block_min
                        if vmin is None
                        else min(vmin, block_min)
                    )
                    vmax = (
                        block_max
                        if vmax is None
                        else max(vmax, block_max)
                    )

                if vmin is None or vmax is None:
                    return False, "", f"No valid pixels in {in_path.name}"

                # A constant corridor contains no internal ranking information.
                # Assigning 50 marks it as neutral while preserving NoData.
                if vmax == vmin:
                    with rasterio.open(out_path, "w", **profile) as dst:
                        for _, window in src.block_windows(1):
                            ma = src.read(
                                1,
                                window=window,
                                masked=True,
                            )
                            mask = np.ma.getmaskarray(ma)
                            data = np.asarray(ma.data)
                            valid = (~mask) & np.isfinite(data)

                            tile = np.full(
                                ma.shape,
                                PERCENTILE_NODATA,
                                dtype=np.float32,
                            )
                            tile[valid] = np.float32(50.0)
                            dst.write(tile, 1, window=window)

                    return (
                        True,
                        f"Constant corridor -> wrote 50.0 to "
                        f"{out_path.name}",
                        "",
                    )

                # Pass 2: construct a high-resolution global histogram.
                edges = np.linspace(
                    vmin,
                    vmax,
                    bins + 1,
                    dtype=np.float64,
                )
                counts = np.zeros(bins, dtype=np.int64)

                for _, window in src.block_windows(1):
                    values = _valid_values(
                        src.read(1, window=window, masked=True)
                    )
                    if values.size == 0:
                        continue

                    np.clip(values, vmin, vmax, out=values)
                    block_counts, _ = np.histogram(
                        values,
                        bins=edges,
                    )
                    counts += block_counts

                total = int(counts.sum())
                if total == 0:
                    return (
                        False,
                        "",
                        f"No valid pixels counted in {in_path.name}",
                    )

                cumulative = np.cumsum(counts, dtype=np.int64)
                cumulative_before = np.zeros_like(cumulative)
                cumulative_before[1:] = cumulative[:-1]

                # Pass 3: map each value to its interpolated percentile rank.
                with rasterio.open(out_path, "w", **profile) as dst:
                    for _, window in src.block_windows(1):
                        ma = src.read(
                            1,
                            window=window,
                            masked=True,
                        )
                        mask = np.ma.getmaskarray(ma)
                        data = np.asarray(
                            ma.data,
                            dtype=np.float64,
                        )
                        valid = (~mask) & np.isfinite(data)

                        tile = np.full(
                            ma.shape,
                            PERCENTILE_NODATA,
                            dtype=np.float32,
                        )

                        if valid.any():
                            values = data[valid].copy()
                            np.clip(
                                values,
                                vmin,
                                vmax,
                                out=values,
                            )

                            # Match NumPy histogram bin assignment. vmax is
                            # explicitly retained in the final bin.
                            idx = (
                                np.searchsorted(
                                    edges,
                                    values,
                                    side="right",
                                )
                                - 1
                            )
                            idx = np.clip(idx, 0, bins - 1)

                            lower = edges[idx]
                            upper = edges[idx + 1]
                            widths = upper - lower

                            # Use the actual value position inside the bin.
                            # Identical values therefore receive identical ranks.
                            fraction = np.divide(
                                values - lower,
                                widths,
                                out=np.zeros_like(values),
                                where=widths > 0,
                            )
                            np.clip(
                                fraction,
                                0.0,
                                1.0,
                                out=fraction,
                            )

                            ranks = (
                                100.0
                                * (
                                    cumulative_before[idx]
                                    + fraction * counts[idx]
                                )
                                / total
                            )

                            tile[valid] = ranks.astype(
                                np.float32,
                                copy=False,
                            )

                        dst.write(tile, 1, window=window)

        return True, f"Wrote {out_path.name}", ""

    except Exception as e:
        # Remove incomplete outputs so that they cannot be mistaken for
        # successfully normalised corridors.
        try:
            if out_path.exists():
                out_path.unlink()
        except OSError:
            pass

        return False, "", f"{in_path.name}: {e}"


def normalize_all_to_percentile_rank(corridor_files):
    """Normalise all pairwise corridors concurrently."""
    corridor_files = [Path(path) for path in corridor_files]

    if not corridor_files:
        raise RuntimeError(
            "No corridor rasters were supplied for normalisation."
        )

    workers = min(
        PERCENTILE_WORKERS,
        len(corridor_files),
    )

    print(
        f"Generating percentile-rank corridor raster(s) with "
        f"{workers} worker(s) and "
        f"{PERCENTILE_BINS:,} histogram bins ..."
    )

    results = []

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = []

        for in_path in corridor_files:
            out_path = in_path.with_name(
                in_path.stem + "_prank.tif"
            )
            futures.append(
                executor.submit(
                    percentile_rank_rasterio,
                    in_path,
                    out_path,
                    PERCENTILE_BINS,
                )
            )

        for future in as_completed(futures):
            results.append(future.result())

    successful = sum(
        success for success, _, _ in results
    )
    failed = len(results) - successful

    for success, stdout, stderr in results:
        if success and stdout:
            print(f"✓ {stdout}")
        elif not success:
            print(f"✗ {stderr}")

    print(
        f"\nPercentile-rank normalisation complete. "
        f"Successful: {successful} | Failed: {failed}"
    )
    print(f"Directory: {out_dir}")

    if failed:
        raise RuntimeError(
            f"{failed} corridor raster(s) failed normalisation. "
            "The network composite was not started."
        )


# Run the normalisation.
normalize_all_to_percentile_rank(corr_files)


In [ ]:
# Step 8: Join all normalised corridors using one minimum composite

# This cell additionally writes:
#   1. a source-index raster identifying which corridor supplied the minimum;
#   2. a CSV relating source indices to corridor filenames;
#   3. support rasters recording the proportion of corridors in which each
#      cell falls within the 1st, 5th and 10th percentiles.

import csv
import os
import platform
import shutil
import subprocess
import tempfile
from pathlib import Path


try:
    GRASS_BIN
except NameError:
    raise RuntimeError(
        "GRASS_BIN is not defined. Run the GRASS setup cell first."
    )


# Supplementary support thresholds. These do not replace the minimum network.
SUPPORT_PERCENTILES = (1, 5, 10)

# r.series is parallelised internally. Eight threads normally provide a good
# balance for large rasters. Reduce these values if memory or disk bandwidth
# becomes limiting.
RSERIES_NPROCS = max(
    1,
    min(8, os.cpu_count() or 1),
)
RSERIES_MEMORY_MB = 512

IS_WINDOWS = platform.system().lower().startswith("win")


def win_short_path(path_string: str) -> str:
    """Return a Windows 8.3 path when available."""
    if not IS_WINDOWS:
        return path_string

    import ctypes

    buffer = ctypes.create_unicode_buffer(32768)
    result = ctypes.windll.kernel32.GetShortPathNameW(
        path_string,
        buffer,
        len(buffer),
    )
    return buffer.value if result else path_string


def safe_arg_path(path: Path) -> str:
    """Quote a path for the temporary platform-specific command script."""
    path_string = str(Path(path))

    if IS_WINDOWS:
        short = win_short_path(path_string)

        if short != path_string:
            return short

        if any(character in path_string for character in " ()"):
            return f'"{path_string}"'

        return path_string

    return "'" + path_string.replace("'", "'\\''") + "'"


def mapcalc_command(expression: str) -> str:
    """Return an r.mapcalc command with platform-appropriate quoting."""
    if IS_WINDOWS:
        return (
            f'r.mapcalc expression="{expression}" --overwrite'
        )

    return (
        f"r.mapcalc expression='{expression}' --overwrite"
    )


# Use exactly the normalised files generated from corr_files rather than a
# wildcard. This prevents stale outputs from previous runs entering the model.
norm_files = [
    Path(path).with_name(
        Path(path).stem + "_prank.tif"
    )
    for path in corr_files
]

missing_norm_files = [
    path for path in norm_files
    if not path.exists()
]

if missing_norm_files:
    missing_names = "\n  ".join(
        path.name for path in missing_norm_files
    )
    raise RuntimeError(
        "The following normalised corridors are missing:\n  "
        + missing_names
    )


for threshold in SUPPORT_PERCENTILES:
    if not 0 < threshold <= 100:
        raise ValueError(
            "Support percentiles must be greater than 0 "
            "and at most 100."
        )


# GRASS raster names and output paths.
raster_names = [
    f"corridor_{index:05d}"
    for index in range(len(norm_files))
]
n_corridors = len(norm_files)

out_min_tif = (
    Path(out_dir)
    / f"{base}_corridors_min.tif"
)
out_source_tif = (
    Path(out_dir)
    / f"{base}_corridors_min_source.tif"
)
out_lookup_csv = (
    Path(out_dir)
    / f"{base}_corridors_min_source_lookup.csv"
)

support_outputs = {
    threshold: (
        Path(out_dir)
        / f"{base}_corridors_support_p{threshold:02d}.tif"
    )
    for threshold in SUPPORT_PERCENTILES
}


# Use a script and an r.series input file rather than one very long command.
# This remains reliable when hundreds of corridors are supplied.
temporary_directory = Path(
    tempfile.mkdtemp(prefix="m2prm_network_")
)
series_file = (
    temporary_directory
    / "normalised_corridors.txt"
)
series_file.write_text(
    "\n".join(raster_names) + "\n",
    encoding="utf-8",
)

commands = []


# Import the percentile-ranked GeoTIFFs into one temporary GRASS location.
for raster_name, tif_path in zip(
    raster_names,
    norm_files,
):
    commands.append(
        f"r.in.gdal "
        f"input={safe_arg_path(tif_path.resolve())} "
        f"output={raster_name} --overwrite"
    )

commands.append(
    f"g.region raster={raster_names[0]}"
)


# r.series -z is slower but avoids open-file limits for exceptionally large
# corridor collections.
close_files_flag = (
    "-z "
    if n_corridors >= 500
    else ""
)
series_argument = safe_arg_path(
    series_file.resolve()
)


# Principal network and corridor-provenance raster in one r.series pass.
commands.append(
    f"r.series {close_files_flag}"
    f"file={series_argument} "
    f"output=min_corridors,min_corridor_source "
    f"method=minimum,min_raster "
    f"nprocs={RSERIES_NPROCS} "
    f"memory={RSERIES_MEMORY_MB} "
    f"--overwrite"
)


# Supplementary recurrence/support products.
#
# Each output is a proportion from 0 to 1, making it comparable when different
# models contain different numbers of corridor pairs.
for threshold in SUPPORT_PERCENTILES:
    tag = f"p{threshold:02d}"
    count_name = f"support_count_{tag}"
    proportion_name = f"support_{tag}"

    commands.append(
        f"r.series {close_files_flag}"
        f"file={series_argument} "
        f"output={count_name} "
        f"method=count "
        f"range=0,{threshold} "
        f"nprocs={RSERIES_NPROCS} "
        f"memory={RSERIES_MEMORY_MB} "
        f"--overwrite"
    )

    # Cells that are NoData in all corridors remain NoData rather than being
    # assigned a zero-support value.
    commands.append(
        mapcalc_command(
            f"{proportion_name} = "
            f"if(isnull(min_corridors), null(), "
            f"float({count_name}) / float({n_corridors}))"
        )
    )


float_create_options = (
    'createopt="COMPRESS=LZW,TILED=YES,'
    'BIGTIFF=IF_SAFER,PREDICTOR=3"'
)
integer_create_options = (
    'createopt="COMPRESS=LZW,TILED=YES,'
    'BIGTIFF=IF_SAFER,PREDICTOR=2"'
)


# Export the principal minimum network.
commands.append(
    f"r.out.gdal -c "
    f"input=min_corridors "
    f"output={safe_arg_path(out_min_tif.resolve())} "
    f"format=GTiff "
    f"type=Float32 "
    f"nodata=-9999 "
    f"{float_create_options} "
    f"--overwrite"
)


# min_raster returns zero-based input indices; -1 is reserved for NoData.
commands.append(
    f"r.out.gdal -c "
    f"input=min_corridor_source "
    f"output={safe_arg_path(out_source_tif.resolve())} "
    f"format=GTiff "
    f"type=Int32 "
    f"nodata=-1 "
    f"{integer_create_options} "
    f"--overwrite"
)


# Export the supplementary support proportions.
for threshold, output_path in support_outputs.items():
    tag = f"p{threshold:02d}"

    commands.append(
        f"r.out.gdal -c "
        f"input=support_{tag} "
        f"output={safe_arg_path(output_path.resolve())} "
        f"format=GTiff "
        f"type=Float32 "
        f"nodata=-9999 "
        f"{float_create_options} "
        f"--overwrite"
    )


# Create and run a platform-specific command script inside a fresh temporary
# GRASS location. If execution fails, the script is retained for diagnosis.
if IS_WINDOWS:
    script_path = (
        temporary_directory
        / "build_network.bat"
    )
    script_lines = [
        "@echo off",
        "setlocal",
    ]

    for command in commands:
        script_lines.extend(
            [
                command,
                "if errorlevel 1 exit /b 1",
            ]
        )

    script_path.write_text(
        "\r\n".join(script_lines) + "\r\n",
        encoding="utf-8",
    )
    runner = [
        "cmd",
        "/D",
        "/C",
        "call",
        str(script_path),
    ]

else:
    script_path = (
        temporary_directory
        / "build_network.sh"
    )
    script_path.write_text(
        "#!/usr/bin/env bash\n"
        "set -e\n"
        + "\n".join(commands)
        + "\n",
        encoding="utf-8",
    )
    runner = [
        "bash",
        str(script_path),
    ]


try:
    tmp_location = (
        f"EPSG:{epsg}"
        if epsg
        else "XY"
    )
except NameError:
    tmp_location = "XY"


print(
    f"Joining {n_corridors} normalised corridor raster(s) "
    f"with r.series using {RSERIES_NPROCS} thread(s) ..."
)

process = subprocess.run(
    [
        GRASS_BIN,
        "--tmp-location",
        tmp_location,
        "--exec",
    ]
    + runner,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)


expected_outputs = [
    out_min_tif,
    out_source_tif,
    *support_outputs.values(),
]

success = (
    process.returncode == 0
    and all(
        path.exists()
        for path in expected_outputs
    )
)


if success:
    # Store the zero-based source-raster lookup next to the provenance raster.
    with out_lookup_csv.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(
            [
                "source_id",
                "corridor_file",
            ]
        )

        for source_id, tif_path in enumerate(norm_files):
            writer.writerow(
                [
                    source_id,
                    tif_path.name,
                ]
            )

    shutil.rmtree(
        temporary_directory,
        ignore_errors=True,
    )

    print(
        f"✓ Minimum corridor network: "
        f"{out_min_tif}"
    )
    print(
        f"✓ Minimum-source raster: "
        f"{out_source_tif}"
    )
    print(
        f"✓ Source lookup: "
        f"{out_lookup_csv}"
    )

    for threshold, output_path in support_outputs.items():
        print(
            f"✓ Support proportion at percentile "
            f"<= {threshold}: {output_path}"
        )

else:
    print(
        "✗ Failed to build the combined corridor network."
    )
    print(
        "STDOUT:\n",
        process.stdout,
    )
    print(
        "STDERR:\n",
        process.stderr,
    )
    print(
        f"Temporary command files retained in: "
        f"{temporary_directory}"
    )

    raise RuntimeError(
        "GRASS corridor-network construction failed."
    )